# Sección E — Transformación de datos (incisos 21 a 24)
**Responsable:** Santiago  
**Propósito:** dejar `Millaje`, `Motor` y `Potencia_Máxima` como dimensiones numéricas sin unidades, expresar en unidades comunes las dimensiones cuantitativas que provienen de CarDekho y de UCI Automobile, y redondear las tres primeras al entero más cercano.

In [1]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Importación de librerías y carga del dataset limpio
#              (resultado de la Sección D).
# Insumo:      data/processed/cardekho_limpio.csv
# Resultado:   DataFrame df.
# ===========================================================
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RUTA_ENTRADA = "data/processed/cardekho_limpio.csv"

df = pd.read_csv(RUTA_ENTRADA, encoding="utf-8-sig")
print(df.shape)

(15244, 36)


## E.21 Eliminación de caracteres y unidades

In [2]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Revisión de Millaje, Motor y Potencia_Máxima leídas como
#              texto, para detectar caracteres o unidades (inciso 21).
# Insumo:      data/processed/cardekho_limpio.csv, df
# Resultado:   DataFrame texto y tabla diagnostico.
# ===========================================================
cols_medidas = ["Millaje", "Motor", "Potencia_Máxima"]

# Torque no existe en esta versión de CarDekho (observación 2 del equipo)
print("¿Existe 'Torque' en el dataset?:", "Torque" in df.columns)

# dtype=str lee cada valor tal como está escrito en el archivo, sin convertirlo
texto = pd.read_csv(RUTA_ENTRADA, encoding="utf-8-sig", dtype=str, usecols=cols_medidas)

# Un valor es "solo número" si tiene dígitos y, si acaso, un punto decimal (796, 19.7)
PATRON_NUMERO = r"-?\d+(?:\.\d+)?"
solo_numero = texto.apply(lambda s: s.str.fullmatch(PATRON_NUMERO, na=False))

diagnostico = pd.DataFrame({
    "Tipo de dato": df[cols_medidas].dtypes.astype(str),
    "Valores": texto.count(),
    "Solo número": solo_numero.sum(),
    "Con caracteres o unidades": (~solo_numero).sum(),
    "Ejemplos": texto.apply(lambda s: ", ".join(s.unique()[:4])),
})
diagnostico

¿Existe 'Torque' en el dataset?: False


,Tipo de dato,Valores,Solo número,Con caracteres o unidades,Ejemplos
Millaje,float64,15244,15244,0,"19.7, 18.9, 17.0, 20.92"
Motor,int64,15244,15244,0,"796, 1197, 998, 1498"
Potencia_Máxima,float64,15244,15244,0,"46.3, 82.0, 80.0, 67.1"


In [3]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Regla de limpieza: se conserva el número y se elimina la
#              unidad o cualquier otro carácter que lo acompañe (inciso 21).
# Insumo:      texto, PATRON_NUMERO
# Resultado:   Función quitar_unidades y DataFrame sin_unidades.
# ===========================================================
def quitar_unidades(serie):
    """Regresa solo el número con el que empieza cada valor ('18.9 kmpl' -> '18.9')."""
    # ^ indica el inicio del texto; lo que sigue al número (unidad, espacios o
    # símbolos) no entra en el paréntesis y se descarta
    return serie.str.strip().str.extract("^(" + PATRON_NUMERO + ")", expand=False)


# Prueba de la regla con valores de ejemplo (NO son del dataset): así se verían
# las medidas si incluyeran su unidad
prueba = pd.Series(["18.9 kmpl", "1197 CC", "90 bhp"])
print(pd.DataFrame({"Valor con unidad": prueba, "Sin unidad": quitar_unidades(prueba)}), "\n")

# Aplicación de la regla a las tres dimensiones del dataset
sin_unidades = texto.apply(quitar_unidades)

print(f"Valores que quedaron igual al aplicar la regla (de {len(texto)} por dimensión):")
print((sin_unidades == texto).sum())

  Valor con unidad Sin unidad
0        18.9 kmpl       18.9
1          1197 CC       1197
2           90 bhp         90 

Valores que quedaron igual al aplicar la regla (de 15244 por dimensión):
Millaje            15244
Motor              15244
Potencia_Máxima    15244
dtype: int64


### Justificación
Eliminación de caracteres y unidades (inciso 21): en esta versión de CarDekho, `Millaje`, `Motor` y `Potencia_Máxima` ya vienen sin unidades, como se había identificado en la Sección A. Al leerlas como texto, los 15,244 valores de cada dimensión son solo número. Aun así se dejó aplicada la regla de limpieza para que el paso sea reproducible: conserva el número y descarta la unidad, como muestra la prueba, y no modificó ningún valor del dataset. `Torque` no existe en esta versión ni puede obtenerse de otra dimensión (observación 2), por lo que los incisos 21 a 24 se aplican a las otras tres dimensiones.

## E.22 Conversión a tipo de dato numérico

In [4]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Conversión de las tres dimensiones, ya sin unidades, a un
#              tipo de dato numérico (inciso 22).
# Insumo:      sin_unidades, df
# Resultado:   Millaje, Motor y Potencia_Máxima numéricas en df.
# ===========================================================
# to_numeric asigna int64 si todos los valores son enteros y float64 si hay
# decimales; errors="coerce" dejaría como NaN cualquier valor que no fuera número
convertidas = sin_unidades.apply(pd.to_numeric, errors="coerce")

print("Nulos generados por la conversión:", convertidas.isnull().sum().sum())
print("¿Mismos valores que ya tenía df? :", convertidas.equals(df[cols_medidas]))

df[cols_medidas] = convertidas

pd.DataFrame({
    "Tipo como texto": sin_unidades.dtypes.astype(str),
    "Tipo numérico": df[cols_medidas].dtypes.astype(str),
    "Mínimo": df[cols_medidas].min(),
    "Media": df[cols_medidas].mean().round(2),
    "Máximo": df[cols_medidas].max(),
})

Nulos generados por la conversión: 0
¿Mismos valores que ya tenía df? : True


,Tipo como texto,Tipo numérico,Mínimo,Media,Máximo
Millaje,str,float64,4.0,19.70,33.54
Motor,str,int64,793.0,1486.17,6592.00
Potencia_Máxima,str,float64,38.4,100.61,626.00


### Justificación
Conversión a numérico (inciso 22): las tres dimensiones se convirtieron con `pd.to_numeric`. `Millaje` (km/l) y `Potencia_Máxima` (bhp) quedaron como `float64` porque tienen decimales, y `Motor` (cc) como `int64` porque todos sus valores son enteros. La conversión no generó nulos y dio exactamente los mismos valores que ya tenía el dataset, por lo que no se perdió información. Con estos tipos ya se pueden hacer operaciones numéricas, como el mínimo, la media y el máximo de la tabla.

## E.23 Unificación de unidades entre fuentes

In [5]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Unidad original, unidad de referencia y factor de conversión
#              de cada dimensión cuantitativa (inciso 23).
# Insumo:      Diccionarios de dimensiones de la Sección A (Tablas A.3 y A.4)
# Resultado:   DataFrame unidades.
# ===========================================================
# Equivalencias exactas entre unidades
MILLA_KM = 1.609344      # 1 milla = 1.609344 km
GALON_L = 3.785411784    # 1 galón de EE. UU. = 3.785411784 l
PULGADA_MM = 25.4        # 1 pulgada = 25.4 mm
LIBRA_KG = 0.45359237    # 1 libra = 0.45359237 kg

MPG_A_KML = MILLA_KM / GALON_L       # millas por galón -> km por litro
IN3_A_CC = (PULGADA_MM / 10) ** 3    # pulgadas cúbicas -> cc (1 in = 2.54 cm)

# CarDekho es el dataset principal: sus unidades se toman como referencia.
# Factor 1 significa que la dimensión ya está en la unidad de referencia.
unidades = pd.DataFrame([
    ["Kilómetros",          "CarDekho", "km",           "km",           1],
    ["Millaje",             "CarDekho", "km/l",         "km/l",         1],
    ["Motor",               "CarDekho", "cc",           "cc",           1],
    ["Potencia_Máxima",     "CarDekho", "bhp",          "bhp",          1],
    ["Asientos",            "CarDekho", "asientos",     "asientos",     1],
    ["Precio_Venta",        "CarDekho", "INR",          "INR",          1],
    ["Año",                 "CarDekho", "año",          "año",          1],
    ["MPG_Ciudad",          "UCI",      "mpg",          "km/l",         MPG_A_KML],
    ["MPG_Carretera",       "UCI",      "mpg",          "km/l",         MPG_A_KML],
    ["Tamano_Motor",        "UCI",      "in³",          "cc",           IN3_A_CC],
    ["Caballos_Fuerza",     "UCI",      "hp",           "bhp",          1],
    ["Peso_Vacio",          "UCI",      "lb",           "kg",           LIBRA_KG],
    ["Altura",              "UCI",      "in",           "mm",           PULGADA_MM],
    ["Ancho",               "UCI",      "in",           "mm",           PULGADA_MM],
    ["Longitud",            "UCI",      "in",           "mm",           PULGADA_MM],
    ["Distancia_Ejes",      "UCI",      "in",           "mm",           PULGADA_MM],
    ["Diametro_Cilindro",   "UCI",      "in",           "mm",           PULGADA_MM],
    ["Carrera_Piston",      "UCI",      "in",           "mm",           PULGADA_MM],
    ["RPM_Maximas",         "UCI",      "rpm",          "rpm",          1],
    ["Relacion_Compresion", "UCI",      "adimensional", "adimensional", 1],
    ["Num_Cilindros",       "UCI",      "cilindros",    "cilindros",    1],
    ["Num_Puertas",         "UCI",      "puertas",      "puertas",      1],
], columns=["Dimensión", "Fuente", "Unidad original", "Unidad de referencia", "Factor"])

unidades["Transformación"] = ["Ninguna" if f == 1 else f"× {f:.8g}" for f in unidades["Factor"]]

# Comprobación: la tabla cubre todas las dimensiones cuantitativas del dataset
# (Tiene_Datos_UCI es un indicador 0/1, no una medida)
cuantitativas = df.select_dtypes(include="number").columns.drop("Tiene_Datos_UCI")
assert set(unidades["Dimensión"]) == set(cuantitativas)

unidades.drop(columns="Factor")

,Dimensión,Fuente,Unidad original,Unidad de referencia,Transformación
0,Kilómetros,CarDekho,km,km,Ninguna
1,Millaje,CarDekho,km/l,km/l,Ninguna
2,Motor,CarDekho,cc,cc,Ninguna
3,Potencia_Máxima,CarDekho,bhp,bhp,Ninguna
4,Asientos,CarDekho,asientos,asientos,Ninguna
5,Precio_Venta,CarDekho,INR,INR,Ninguna
6,Año,CarDekho,año,año,Ninguna
7,MPG_Ciudad,UCI,mpg,km/l,× 0.42514371
8,MPG_Carretera,UCI,mpg,km/l,× 0.42514371
9,Tamano_Motor,UCI,in³,cc,× 16.387064


In [6]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Comprobación de la unidad de Tamano_Motor (observación 3):
#              se compara con la cilindrada calculada a partir del diámetro,
#              la carrera y el número de cilindros (inciso 23).
# Insumo:      df (registros con datos de UCI, una fila por marca)
# Resultado:   Tabla comprobacion con la razón entre ambas cilindradas.
# ===========================================================
# Cilindrada = (π / 4) × diámetro² × carrera × número de cilindros.
# Con el diámetro y la carrera en pulgadas, el resultado queda en pulgadas cúbicas.
marcas_uci = (df[df["Tiene_Datos_UCI"] == 1]
              .drop_duplicates("Marca_Homologada")
              .set_index("Marca_Homologada"))

cilindrada = ((np.pi / 4) * marcas_uci["Diametro_Cilindro"] ** 2
              * marcas_uci["Carrera_Piston"] * marcas_uci["Num_Cilindros"])

comprobacion = pd.DataFrame({
    "Cilindrada calculada (in³)": cilindrada,
    "Tamano_Motor": marcas_uci["Tamano_Motor"],
    "Razón": cilindrada / marcas_uci["Tamano_Motor"],
}).sort_index()

print("Mediana de la razón:", round(comprobacion["Razón"].median(), 2))
comprobacion.round(2)

Mediana de la razón: 1.0


,Cilindrada calculada (in³),Tamano_Motor,Razón
Marca_Homologada,,,
audi,130.57,130.71,1.00
bmw,165.11,166.88,0.99
honda,99.31,99.31,1.00
isuzu,101.98,102.50,0.99
jaguar,301.29,280.67,1.07
mercedes benz,227.73,226.50,1.01
nissan,128.62,127.89,1.01
porsche,205.19,187.20,1.10
renault,146.68,132.00,1.11


In [7]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Conversión de las dimensiones de UCI a la unidad de referencia
#              y renombrado de las que llevan la unidad en su nombre (inciso 23).
# Insumo:      df, unidades
# Resultado:   df con las dimensiones de UCI en unidades métricas y tabla evidencia.
# ===========================================================
por_convertir = unidades[unidades["Factor"] != 1]
dims = por_convertir["Dimensión"].tolist()

antes = df[dims].agg(["min", "max"]).T

# Cada dimensión se multiplica por su factor
for dim, factor in zip(dims, por_convertir["Factor"]):
    df[dim] = df[dim] * factor

despues = df[dims].agg(["min", "max"]).T

evidencia = pd.DataFrame({
    "Unidad original": por_convertir["Unidad original"].values,
    "Mín. antes": antes["min"],
    "Máx. antes": antes["max"],
    "Unidad de referencia": por_convertir["Unidad de referencia"].values,
    "Mín. después": despues["min"],
    "Máx. después": despues["max"],
})

# MPG significa "millas por galón": al quedar en km/l se renombran como Millaje
df = df.rename(columns={"MPG_Ciudad": "Millaje_Ciudad", "MPG_Carretera": "Millaje_Carretera"})
print("Renombradas: MPG_Ciudad -> Millaje_Ciudad | MPG_Carretera -> Millaje_Carretera")

evidencia.round(2)

Renombradas: MPG_Ciudad -> Millaje_Ciudad | MPG_Carretera -> Millaje_Carretera


,Unidad original,Mín. antes,Máx. antes,Unidad de referencia,Mín. después,Máx. después
MPG_Ciudad,mpg,14.33,31.00,km/l,6.09,13.18
MPG_Carretera,mpg,18.33,36.00,km/l,7.79,15.31
Tamano_Motor,in³,99.31,280.67,cc,1627.36,4599.30
Peso_Vacio,lb,2096.77,4027.33,kg,951.08,1826.77
Altura,in,51.10,56.24,mm,1297.94,1428.40
Ancho,in,63.55,71.06,mm,1614.17,1804.99
Longitud,in,160.77,196.97,mm,4083.54,5002.95
Distancia_Ejes,in,92.28,110.92,mm,2343.91,2817.50
Diametro_Cilindro,in,3.02,3.82,mm,76.75,97.03
Carrera_Piston,in,2.98,3.90,mm,75.79,99.06


In [8]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Comparación de las dimensiones equivalentes de ambas fuentes,
#              ya en la misma unidad (inciso 23).
# Insumo:      df
# Resultado:   DataFrame comparacion.
# ===========================================================
# De UCI solo se usan los registros con datos reales (sin los valores imputados)
con_uci = df[df["Tiene_Datos_UCI"] == 1]

pares = [("Motor", "Tamano_Motor", "cc"),
         ("Potencia_Máxima", "Caballos_Fuerza", "bhp"),
         ("Millaje", "Millaje_Ciudad", "km/l"),
         ("Millaje", "Millaje_Carretera", "km/l")]
filas = []
for col_cd, col_uci, unidad in pares:
    filas.append([f"{col_cd} / {col_uci}", unidad,
                  df[col_cd].min(), df[col_cd].median(), df[col_cd].max(),
                  con_uci[col_uci].min(), con_uci[col_uci].median(), con_uci[col_uci].max()])
comparacion = pd.DataFrame(filas, columns=["Par", "Unidad", "CD mín", "CD mediana", "CD máx",
                                           "UCI mín", "UCI mediana", "UCI máx"])
comparacion.round(1)

,Par,Unidad,CD mín,CD mediana,CD máx,UCI mín,UCI mediana,UCI máx
0,Motor / Tamano_Motor,cc,793.0,1248.0,6592.0,1627.4,1947.0,4599.3
1,Potencia_Máxima / Caballos_Fuerza,bhp,38.4,88.5,626.0,77.0,81.1,210.4
2,Millaje / Millaje_Ciudad,km/l,4.0,19.7,33.5,6.1,11.7,13.2
3,Millaje / Millaje_Carretera,km/l,4.0,19.7,33.5,7.8,14.0,15.3


In [9]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Revisión de Millaje por tipo de combustible: en los vehículos
#              a gas el rendimiento se publica en km/kg (inciso 23).
# Insumo:      df
# Resultado:   Tabla de Millaje por combustible y registros a gas.
# ===========================================================
a_gas = df["Combustible"].isin(["CNG", "LPG"])
print(f"Registros a gas (CNG y LPG): {a_gas.sum()} de {len(df)} ({a_gas.mean() * 100:.2f}%)")

df.groupby("Combustible")["Millaje"].agg(["count", "min", "median", "max"])

Registros a gas (CNG y LPG): 343 de 15244 (2.25%)


,count,min,median,max
Combustible,,,,
CNG,299,11.88,26.60,33.54
Diesel,7342,7.50,20.50,28.40
Electric,4,19.16,19.16,19.16
LPG,44,13.45,17.30,26.20
Petrol,7555,4.00,18.90,28.09


### Justificación
- Unidad de referencia: CarDekho es el dataset principal y usa unidades métricas (km, km/l, cc), por lo que sus unidades se tomaron como referencia y sus dimensiones no se modificaron.
- Conversión de UCI: sus dimensiones estaban en las unidades que se usan en Estados Unidos (millas por galón, pulgadas y libras) y se convirtieron con equivalencias exactas: mpg a km/l (× 0.425144, porque 1 milla = 1.609344 km y 1 galón = 3.785412 l), in³ a cc (× 16.387064), pulgadas a mm (× 25.4) y libras a kg (× 0.453592). `MPG_Ciudad` y `MPG_Carretera` se renombraron a `Millaje_Ciudad` y `Millaje_Carretera`, porque su nombre indicaba una unidad que ya no tienen.
- Unidad de `Tamano_Motor`: UCI no la documenta (observación 3). La cilindrada calculada con el diámetro, la carrera y el número de cilindros coincide con `Tamano_Motor` (mediana de la razón: 1.00), lo que confirma que está en pulgadas cúbicas; si estuviera en cc, la razón sería cercana a 16.
- Potencia: `Caballos_Fuerza` (hp) y `Potencia_Máxima` (bhp) están en la misma unidad, el caballo de fuerza. bhp (*brake horsepower*) solo indica que la potencia se midió en el eje del motor, así que no se requiere conversión.
- `Millaje` en vehículos a gas: según el diccionario de la Sección A, en los 343 registros de CNG y LPG (2.25 %) el rendimiento está en km/kg. No se convirtieron a km/l porque pasar de kilogramos a litros depende de la densidad del combustible, que no está en los datos. Se conservan sus valores y debe tenerse en cuenta al comparar `Millaje` entre combustibles.
- Precio: `Precio` (UCI, en USD) se eliminó en el inciso 18, por lo que no hay monedas que unificar. `Precio_Venta` se mantiene en INR.

## E.24 Redondeo al entero más cercano

In [10]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Redondeo de Millaje, Motor y Potencia_Máxima al entero más
#              cercano (inciso 24).
# Insumo:      df, cols_medidas
# Resultado:   Las tres dimensiones como enteros (int64) y tabla resumen.
# ===========================================================
sin_redondear = df[cols_medidas].copy()

# Valores que terminan exactamente en .5: están a la misma distancia de dos enteros
empates = (sin_redondear % 1 == 0.5).sum()

# round() deja el entero más cercano; como ya no hay decimales se guarda como int64
df[cols_medidas] = df[cols_medidas].round().astype("int64")

resumen = pd.DataFrame({
    "Tipo antes": sin_redondear.dtypes.astype(str),
    "Tipo después": df[cols_medidas].dtypes.astype(str),
    "Valores modificados": (sin_redondear != df[cols_medidas]).sum(),
    "Empates en .5": empates,
    "Media antes": sin_redondear.mean(),
    "Media después": df[cols_medidas].mean(),
    "Mediana antes": sin_redondear.median(),
    "Mediana después": df[cols_medidas].median(),
})
resumen.round(3)

,Tipo antes,Tipo después,Valores modificados,Empates en .5,Media antes,Media después,Mediana antes,Mediana después
Millaje,float64,int64,13433,715,19.697,19.718,19.67,20.0
Motor,int64,int64,0,0,1486.172,1486.172,1248.00,1248.0
Potencia_Máxima,float64,int64,10657,1092,100.608,100.611,88.50,88.0


In [11]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Ejemplos del redondeo: primeros registros y valores más
#              frecuentes que terminan en .5 (inciso 24).
# Insumo:      sin_redondear, df
# Resultado:   Tablas de ejemplos antes y después.
# ===========================================================
ejemplos = pd.DataFrame({
    "Nombre": df["Nombre"],
    "Millaje antes": sin_redondear["Millaje"],
    "Millaje": df["Millaje"],
    "Motor antes": sin_redondear["Motor"],
    "Motor": df["Motor"],
    "Potencia_Máxima antes": sin_redondear["Potencia_Máxima"],
    "Potencia_Máxima": df["Potencia_Máxima"],
})
print(ejemplos.head(5), "\n")

# Empates: los cuatro valores terminados en .5 más frecuentes de Potencia_Máxima
potencia = sin_redondear["Potencia_Máxima"]
frecuentes = potencia[potencia % 1 == 0.5].value_counts().head(4)
pd.DataFrame({"Potencia_Máxima antes": frecuentes.index,
              "Registros": frecuentes.values,
              "Potencia_Máxima": frecuentes.index.to_series().round().astype("int64").values})

          Nombre  Millaje antes  Millaje  Motor antes  Motor  Potencia_Máxima antes  Potencia_Máxima
0    Maruti Alto          19.70       20          796    796                  46.30               46
1  Hyundai Grand          18.90       19         1197   1197                  82.00               82
2    Hyundai i20          17.00       17         1197   1197                  80.00               80
3    Maruti Alto          20.92       21          998    998                  67.10               67
4  Ford Ecosport          22.77       23         1498   1498                  98.59               99 



,Potencia_Máxima antes,Registros,Potencia_Máxima
0,88.5,581,88
1,103.5,100,104
2,108.5,93,108
3,168.5,76,168


### Justificación
Redondeo (inciso 24): las tres dimensiones se redondearon con `round()` y se guardaron como `int64`, porque después del redondeo ya no tienen decimales. Se modificaron 13,433 valores de `Millaje` y 10,657 de `Potencia_Máxima`; `Motor` no cambió porque sus valores ya eran enteros. Cada valor cambia como máximo 0.5 unidades.

Los valores que terminan exactamente en .5 están a la misma distancia de dos enteros. En ese caso `round()` de pandas elige el entero par (88.5 queda en 88 y 103.5 en 104). Son 715 valores en `Millaje` y 1,092 en `Potencia_Máxima`. Este criterio evita que todos los empates suban, lo que desplazaría la media hacia arriba; con él, la media de `Potencia_Máxima` pasó de 100.608 a 100.611.

## Comprobación y exportación

In [12]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Comprobación de las transformaciones de los incisos 21 a 24:
#              nulos, tipos de dato, rangos y tamaño del dataset.
# Insumo:      df
# Resultado:   Reporte de comprobaciones.
# ===========================================================
# 1) Nulos: las transformaciones no deben generar ninguno
print("1) Nulos totales                :", df.isnull().sum().sum())

# 2) Tipos de dato de las tres dimensiones transformadas
print("2) Tipos de dato                :", df[cols_medidas].dtypes.astype(str).to_dict())

# 3) Rangos: no debe haber valores negativos ni iguales a cero
print("3) Valores menores o iguales a 0:", (df[cols_medidas] <= 0).sum().sum())
print(df[cols_medidas].agg(["min", "max"]), "\n")

# 4) El dataset conserva sus observaciones y sus dimensiones
print("4) Dimensiones del dataset      :", df.shape)

# 5) Registros que quedaron idénticos a otro después del redondeo
print("5) Registros idénticos a otro   :", df.duplicated().sum())

1) Nulos totales                : 0
2) Tipos de dato                : {'Millaje': 'int64', 'Motor': 'int64', 'Potencia_Máxima': 'int64'}
3) Valores menores o iguales a 0: 0
     Millaje  Motor  Potencia_Máxima
min        4    793               38
max       34   6592              626 

4) Dimensiones del dataset      : (15244, 36)
5) Registros idénticos a otro   : 6


### Justificación
Las transformaciones no generaron nulos ni valores inválidos, y el dataset conserva sus 15,244 observaciones y 36 dimensiones. El redondeo dejó 6 registros idénticos a otro: antes solo se distinguían por los decimales de `Potencia_Máxima` (por ejemplo, 68.00 y 68.05). No se eliminaron, porque eran registros distintos antes de redondear y la eliminación de duplicados corresponde al inciso 17.

In [13]:
# ===========================================================
# Autor:       Santiago
# Fecha:       05 octubre 2026
# Descripción: Exportación del dataset transformado para los incisos
#              siguientes de la Sección E y para las Secciones F y G.
# Insumo:      df
# Resultado:   data/processed/cardekho_transformado.csv
# ===========================================================
df.to_csv("data/processed/cardekho_transformado.csv", index=False, encoding="utf-8-sig")
print("Exportado:", df.shape)

Exportado: (15244, 36)
